# LeetCode 881 — Boats to Save People bằng Python

Link đề: https://leetcode.com/problems/boats-to-save-people/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Two Pointers, Sorting

Skeleton LeetCode:

```python
class Solution:
    def numRescueBoats(self, people: List[int], limit: int) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: sắp xếp rồi dùng **hai con trỏ**: người **nặng nhất** luôn chiếm một thuyền; nếu **nhẹ nhất** ghép được thì ghép. Chứng minh bằng **lập luận đổi chỗ** (exchange argument). Oracle: **vét cạn mọi cách ghép cặp** (mọi phân hoạch thành nhóm tối đa 2 người) trên dữ liệu nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi thuyền chở **tối đa 2 người** và tổng cân nặng `≤ limit`; tìm số thuyền **ít nhất** để chở hết.
2. Nắm công cụ Python: `sort`, hai con trỏ `lo`/`hi`, mảng đếm (counting sort), tìm kiếm nhị phân trên đáp án, `itertools` để vét cạn, bitmask quy hoạch động.
3. Đi từ **quy hoạch động bitmask (brute force)** → **sort + hai con trỏ** → **counting sort + hai con trỏ trên bảng đếm** → **nhị phân trên số cặp**.
4. Hiểu mấu chốt: số thuyền `= n − (số cặp tối đa)`; người nặng nhất `hi` chỉ có thể ghép với **người nhẹ nhất** còn lại `lo` (nếu `lo` không ghép được với `hi` thì không ai ghép được) nên cứ xét cặp `(lo, hi)`.
5. Tránh lỗi kinh điển: ghép các phần tử **liền kề** sau sắp xếp, quên sắp xếp, vòng lặp `while lo < hi` bỏ sót người đứng giữa, dùng `<` thay vì `≤` với `limit`, và cho phép 3 người một thuyền.

## 2. Đề bài & ví dụ

Mảng `people[i]` là cân nặng của người thứ `i`. Có **vô hạn** thuyền, mỗi thuyền chở tối đa tổng `limit`. Mỗi thuyền chở **tối đa hai người** cùng lúc, miễn tổng cân nặng `≤ limit`.

Trả về **số thuyền tối thiểu** để chở mọi người.

**Ví dụ 1:** `people = [1,2]`, `limit = 3` → `1` (cả hai cùng một thuyền).

**Ví dụ 2:** `people = [3,2,2,1]`, `limit = 3` → `3` (`(1,2)`, `(2)`, `(3)`).

**Ví dụ 3:** `people = [3,5,3,4]`, `limit = 5` → `4` (không ai ghép được).

**Ràng buộc:** `1 <= people.length <= 5·10^4`; `1 <= people[i] <= limit <= 3·10^4`.

In [1]:
import itertools
import random
import time
from functools import lru_cache
from typing import List

examples = [(([1, 2], 3), 1), (([3, 2, 2, 1], 3), 3), (([3, 5, 3, 4], 5), 4)]
for (p, lim), want in examples:
    print(f"people = {p}, limit = {lim} -> {want}")

people = [1, 2], limit = 3 -> 1
people = [3, 2, 2, 1], limit = 3 -> 3
people = [3, 5, 3, 4], limit = 5 -> 4


## 3. Kiến thức nền

### 3.1 Số thuyền và số cặp

Mỗi thuyền chở 1 hoặc 2 người. Nếu có `m` thuyền chở 2 người thì số thuyền `= m + (n − 2m) = n − m`. Vì vậy **tối thiểu hoá số thuyền ⇔ tối đa hoá số cặp hợp lệ** `m` (mỗi cặp có tổng `≤ limit`, mỗi người thuộc tối đa một cặp).

In [2]:
people, limit = [3, 2, 2, 1], 3
n = len(people)
for m in range(0, n // 2 + 1):
    print(f"m = {m} cặp -> số thuyền = n - m = {n - m}")
print("cặp hợp lệ trong [3,2,2,1] (limit 3):", [(i, j) for i, j in itertools.combinations(range(n), 2) if people[i] + people[j] <= limit])

m = 0 cặp -> số thuyền = n - m = 4
m = 1 cặp -> số thuyền = n - m = 3
m = 2 cặp -> số thuyền = n - m = 2
cặp hợp lệ trong [3,2,2,1] (limit 3): [(1, 3), (2, 3)]


### 3.2 Sắp xếp và hai con trỏ

Sau khi `sort`, `lo` trỏ người nhẹ nhất còn lại, `hi` trỏ người nặng nhất còn lại. Mỗi bước xét cặp `(people[lo], people[hi])`: nếu `people[lo] + people[hi] ≤ limit` thì cùng một thuyền (`lo += 1`), người nặng luôn lên thuyền (`hi -= 1`).

### 3.3 Counting sort

Cân nặng nằm trong `[1, limit]` với `limit ≤ 3·10^4`: có thể đếm số người theo cân nặng bằng mảng `cnt[w]` thay vì sắp xếp `O(n log n)`.

### 3.4 Lập luận đổi chỗ (exchange argument)

Để chứng minh một lựa chọn tham lam đúng: lấy một nghiệm tối ưu bất kỳ **không** dùng lựa chọn đó, rồi **đổi chỗ** để dùng lựa chọn mà không làm nghiệm xấu đi. Ta dùng ở mục 10.

In [3]:
from collections import Counter
cnt = Counter([3, 2, 2, 1])
print(sorted(cnt.items()))

[(1, 1), (2, 2), (3, 1)]


## 4. Ý tưởng nền tảng / trực giác

`people = [3, 2, 2, 1]`, `limit = 3`. Sắp xếp: `[1, 2, 2, 3]`.

- Người nặng nhất `3`: cần một thuyền; ghép với `1` thì `4 > 3` ✗. Không ai nhẹ hơn `1` nên `3` đi một mình.
- Người nặng nhất còn lại `2`: ghép với người nhẹ nhất `1` thì `3 ≤ 3` ✓ → cặp `(1, 2)`.
- Còn lại `2`: đi một mình.

Tổng `3` thuyền.

Nếu người nặng nhất không ghép được với người **nhẹ nhất**, thì với bất kỳ ai khác (nặng hơn hoặc bằng người nhẹ nhất) cũng không ghép được ⇒ nó đi một mình. Nếu ghép được với người nhẹ nhất thì ghép (dùng người nhẹ nhất cho người nặng nhất là "lãng phí ít nhất" khả năng ghép của người khác).

In [4]:
p = sorted([3, 2, 2, 1])
lo, hi, boats = 0, len(p) - 1, 0
while lo <= hi:
    fit = p[lo] + p[hi] <= 3
    print(f"lo={lo} ({p[lo]}), hi={hi} ({p[hi]}): {p[lo]}+{p[hi]}={p[lo] + p[hi]} {'<=' if fit else '>'} 3 -> {'ghép' if fit else 'người nặng đi một mình'}")
    if fit:
        lo += 1
    hi -= 1
    boats += 1
print("số thuyền =", boats)

lo=0 (1), hi=3 (3): 1+3=4 > 3 -> người nặng đi một mình
lo=0 (1), hi=2 (2): 1+2=3 <= 3 -> ghép
lo=1 (2), hi=1 (2): 2+2=4 > 3 -> người nặng đi một mình
số thuyền = 3


## 5. Cách 1 — Brute Force: quy hoạch động bitmask

**Ý tưởng:** biểu diễn tập người **chưa được chở** bằng bitmask. `dp[mask]` = số thuyền ít nhất để chở tập `mask`. Lấy người đầu tiên (bit thấp nhất) trong `mask`: hoặc đi một mình, hoặc ghép với một người khác trong `mask` nếu tổng `≤ limit`. Không giả định tính chất tham lam nào.

In [5]:
def boats_bitmask(people, limit):
    n = len(people)

    @lru_cache(None)
    def dp(mask):
        if mask == 0:
            return 0
        i = (mask & -mask).bit_length() - 1              # người thấp bit nhất còn lại
        rest = mask & (mask - 1)
        best = 1 + dp(rest)                              # đi một mình
        m2 = rest
        while m2:
            j = (m2 & -m2).bit_length() - 1
            m2 &= m2 - 1
            if people[i] + people[j] <= limit:
                best = min(best, 1 + dp(rest & ~(1 << j)))   # ghép i với j
        return best

    return dp((1 << n) - 1)


for (p, lim), want in examples:
    assert boats_bitmask(p, lim) == want
print([boats_bitmask(p, lim) for (p, lim), _ in examples])

[1, 3, 4]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng O(2^n · n): chỉ dùng được cho n nhỏ)
from functools import lru_cache


class SolutionBitmask:
    def numRescueBoats(self, people: List[int], limit: int) -> int:
        n = len(people)

        @lru_cache(None)
        def dp(mask):
            if mask == 0:
                return 0
            i = (mask & -mask).bit_length() - 1
            rest = mask & (mask - 1)
            best = 1 + dp(rest)
            m2 = rest
            while m2:
                j = (m2 & -m2).bit_length() - 1
                m2 &= m2 - 1
                if people[i] + people[j] <= limit:
                    best = min(best, 1 + dp(rest & ~(1 << j)))
            return best

        return dp((1 << n) - 1)

**Độ phức tạp:** `2^n` trạng thái × `O(n)` chuyển → `O(2^n · n)`. Với `n = 5·10^4` là không thể.

**Nút thắt:** ta thử mọi cách ghép trong khi lập luận đổi chỗ chỉ ra lựa chọn tối ưu luôn là cặp `(nhẹ nhất, nặng nhất)`.

## 6. Ý tưởng thuật toán / bước đột phá

Sắp xếp. Xét người nặng nhất `hi`:

- nếu `people[lo] + people[hi] ≤ limit` (`lo` nhẹ nhất): ghép `lo` với `hi`;
- ngược lại `hi` đi một mình.

Mỗi bước loại ít nhất một người ⇒ `O(n)` sau sắp xếp: tổng `O(n log n)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** sort + hai con trỏ, **Cách 3** counting sort + hai con trỏ trên bảng đếm, **Cách 4** nhị phân trên số cặp.

### Cách 2 — Sort + hai con trỏ

**Giải thích từng đoạn:**

1. `people.sort()`; `lo = 0`, `hi = n − 1`, `boats = 0`.
2. Khi `lo ≤ hi`: `boats += 1`; nếu `people[lo] + people[hi] ≤ limit` thì `lo += 1` (người nhẹ ghép cùng); luôn `hi -= 1`.

In [7]:
def boats_two_pointers(people, limit):
    p = sorted(people)
    lo, hi, boats = 0, len(p) - 1, 0
    while lo <= hi:
        if p[lo] + p[hi] <= limit:                       # người nhẹ nhất ghép được với người nặng nhất
            lo += 1
        hi -= 1                                          # người nặng nhất luôn lên thuyền
        boats += 1
    return boats


for (p, lim), want in examples:
    assert boats_two_pointers(p, lim) == want
print([boats_two_pointers(p, lim) for (p, lim), _ in examples])

[1, 3, 4]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionTwoPointers:
    def numRescueBoats(self, people: List[int], limit: int) -> int:
        people.sort()
        lo, hi, boats = 0, len(people) - 1, 0
        while lo <= hi:
            if people[lo] + people[hi] <= limit:
                lo += 1
            hi -= 1
            boats += 1
        return boats

**Độ phức tạp:** `O(n log n)` (sắp xếp) + `O(n)`; bộ nhớ `O(1)` thêm (nếu sắp xếp tại chỗ).

### Cách 3 — Counting sort + hai con trỏ trên bảng đếm

**Ý tưởng:** `cnt[w]` = số người nặng `w`. Hai con trỏ `lo` (từ `1` lên) và `hi` (từ `limit` xuống), bỏ qua các `w` có `cnt = 0`. Với `lo < hi`: nếu `lo + hi ≤ limit` thì ghép `k = min(cnt[lo], cnt[hi])` cặp cùng lúc (`boats += k`, giảm cả hai); ngược lại cả `cnt[hi]` người nặng `hi` đi một mình. Với `lo == hi` (cùng cân nặng `w`): nếu `2w ≤ limit` thì cần `⌈cnt/2⌉` thuyền, không thì `cnt` thuyền.

In [9]:
def boats_counting(people, limit):
    cnt = [0] * (limit + 1)
    for w in people:
        cnt[w] += 1
    lo, hi, boats = 1, limit, 0
    while True:
        while lo <= hi and cnt[lo] == 0:
            lo += 1
        while hi >= lo and cnt[hi] == 0:
            hi -= 1
        if lo > hi:
            break
        if lo == hi:                                     # chỉ còn một cân nặng w
            boats += (cnt[lo] + 1) // 2 if 2 * lo <= limit else cnt[lo]
            break
        if lo + hi <= limit:
            k = min(cnt[lo], cnt[hi])                    # ghép k cặp (nhẹ nhất, nặng nhất)
            boats += k
            cnt[lo] -= k
            cnt[hi] -= k
        else:
            boats += cnt[hi]                             # mọi người nặng hi đi một mình
            cnt[hi] = 0
    return boats


for (p, lim), want in examples:
    assert boats_counting(p, lim) == want
print([boats_counting(p, lim) for (p, lim), _ in examples])

[1, 3, 4]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionCounting:
    def numRescueBoats(self, people: List[int], limit: int) -> int:
        cnt = [0] * (limit + 1)
        for w in people:
            cnt[w] += 1
        lo, hi, boats = 1, limit, 0
        while True:
            while lo <= hi and cnt[lo] == 0:
                lo += 1
            while hi >= lo and cnt[hi] == 0:
                hi -= 1
            if lo > hi:
                break
            if lo == hi:
                boats += (cnt[lo] + 1) // 2 if 2 * lo <= limit else cnt[lo]
                break
            if lo + hi <= limit:
                k = min(cnt[lo], cnt[hi])
                boats += k
                cnt[lo] -= k
                cnt[hi] -= k
            else:
                boats += cnt[hi]
                cnt[hi] = 0
        return boats

**Độ phức tạp:** `O(n + limit)` thời gian, `O(limit)` bộ nhớ. Tốt hơn sắp xếp khi `limit` nhỏ hơn `n log n`.

### Cách 4 — Nhị phân trên số cặp

**Ý tưởng:** gọi `can(m)` = "có thể tạo `m` cặp hợp lệ không". Khi đã sắp xếp, `can(m)` đúng ⇔ trong `2m` người **nhẹ nhất**, ghép **ngoài cùng với trong cùng** `(p[i], p[2m−1−i])` đều có tổng `≤ limit`. `can` đơn điệu (nếu tạo được `m` cặp thì tạo được `m − 1`), nên nhị phân tìm `m` lớn nhất trong `[0, n/2]`; đáp án `n − m`.

In [11]:
def boats_binary_search(people, limit):
    p = sorted(people)
    n = len(p)

    def can(m):                                          # tạo được m cặp? dùng 2m người nhẹ nhất, ghép ngoài với trong
        return all(p[i] + p[2 * m - 1 - i] <= limit for i in range(m))

    lo, hi = 0, n // 2                                   # tìm m lớn nhất thoả can(m)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if can(mid):
            lo = mid
        else:
            hi = mid - 1
    return n - lo


for (p, lim), want in examples:
    assert boats_binary_search(p, lim) == want
print([boats_binary_search(p, lim) for (p, lim), _ in examples])

[1, 3, 4]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionBinary:
    def numRescueBoats(self, people: List[int], limit: int) -> int:
        p = sorted(people)
        n = len(p)

        def can(m):
            return all(p[i] + p[2 * m - 1 - i] <= limit for i in range(m))

        lo, hi = 0, n // 2
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if can(mid):
                lo = mid
            else:
                hi = mid - 1
        return n - lo

**Độ phức tạp:** `O(n log n)` (sắp xếp) + `O(n log n)` (nhị phân, mỗi lần `can` `O(n)`), `O(n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `people = [3,2,2,1]`, `limit = 3`, sau sắp xếp `[1,2,2,3]`:

| bước | `lo` | `hi` | `p[lo] + p[hi]` | quyết định | `boats` |
|---|---|---|---|---|---|
| 1 | 0 | 3 | `1 + 3 = 4 > 3` | `3` đi một mình; `hi = 2` | 1 |
| 2 | 0 | 2 | `1 + 2 = 3 ≤ 3` | ghép `(1, 2)`; `lo = 1`, `hi = 1` | 2 |
| 3 | 1 | 1 | `2 + 2 = 4 > 3` | `2` đi một mình; `hi = 0` | 3 |

`lo = 1 > hi = 0` → dừng, `3` thuyền ✓. Hàm trace:

In [13]:
def trace_two_pointers(people, limit):
    p = sorted(people)
    lo, hi, boats = 0, len(p) - 1, 0
    print(f"sau sắp xếp: {p}, limit = {limit}")
    print(f"{'lo':>3} {'hi':>3} {'p[lo]+p[hi]':>12}  quyết định                     boats")
    while lo <= hi:
        s = p[lo] + p[hi]
        if s <= limit:
            action = f"ghép ({p[lo]},{p[hi]})" if lo != hi else f"{p[hi]} đi một mình"
            lo += 1
        else:
            action = f"{p[hi]} đi một mình"
        boats += 1
        print(f"{lo - (1 if s <= limit else 0):>3} {hi:>3} {s:>12}  {action:<30} {boats}")
        hi -= 1
    print("=> số thuyền =", boats)
    return boats


assert trace_two_pointers([3, 2, 2, 1], 3) == 3
print()
assert trace_two_pointers([3, 5, 3, 4], 5) == 4

sau sắp xếp: [1, 2, 2, 3], limit = 3
 lo  hi  p[lo]+p[hi]  quyết định                     boats
  0   3            4  3 đi một mình                  1
  0   2            3  ghép (1,2)                     2
  1   1            4  2 đi một mình                  3
=> số thuyền = 3

sau sắp xếp: [3, 3, 4, 5], limit = 5
 lo  hi  p[lo]+p[hi]  quyết định                     boats
  0   3            8  5 đi một mình                  1
  0   2            7  4 đi một mình                  2
  0   1            6  3 đi một mình                  3
  0   0            6  3 đi một mình                  4
=> số thuyền = 4


**Trace Cách 4** với `[1,2,2,3]`, `limit = 3`: `n = 4`, `m ∈ [0, 2]`. `mid = 1`: `can(1)`: `p[0] + p[1] = 3 ≤ 3` ✓ → `lo = 1`. `mid = 2`: `can(2)`: `p[0] + p[3] = 4 > 3` ✗ → `hi = 1`. `m = 1` ⇒ `4 − 1 = 3` ✓.

## 9. Code nộp LeetCode

Chọn **Cách 2** (sort + hai con trỏ). Giữ nguyên `class Solution`, `numRescueBoats(self, people, limit)`:

In [14]:
class Solution:
    def numRescueBoats(self, people: List[int], limit: int) -> int:
        people.sort()
        lo, hi, boats = 0, len(people) - 1, 0
        while lo <= hi:
            if people[lo] + people[hi] <= limit:          # người nhẹ nhất ghép được với người nặng nhất
                lo += 1
            hi -= 1                                       # người nặng nhất luôn lên thuyền
            boats += 1
        return boats


print([Solution().numRescueBoats(list(p), lim) for (p, lim), _ in examples])      # [1, 3, 4]
assert all(Solution().numRescueBoats(list(p), lim) == want for (p, lim), want in examples)

[1, 3, 4]


## 10. Vì sao đúng

**(a) Số thuyền `= n − m`** với `m` là số cặp (mục 3.1). Cần chứng minh thuật toán tìm được số cặp **tối đa** `m*`.

**(b) Hai con trỏ cho số cặp tối đa (lập luận đổi chỗ).** Gọi `H` là người nặng nhất, `L` là người nhẹ nhất (`L ≠ H`) trong tập còn lại.

- *Nếu `L + H > limit`*: mọi người `x` có `x ≥ L` nên `x + H > limit`; `H` không ghép được với ai, tức `H` đi một mình trong **mọi** nghiệm. Bỏ `H` đi không đổi `m*`.
- *Nếu `L + H ≤ limit`*: có một nghiệm tối ưu chứa cặp `{L, H}`. Lấy nghiệm tối ưu `M` bất kỳ (tập cặp rời nhau, kích thước `m*`):
  - `L`, `H` đã là một cặp: xong.
  - `H` ghép với `y`, `L` ghép với `z` (`y ≠ L`): thay hai cặp `{H, y}`, `{L, z}` bằng `{H, L}` và `{y, z}`. Cặp `{H, L}` hợp lệ theo giả thiết. Cặp `{y, z}` hợp lệ vì `z ≤ H` (H nặng nhất) nên `y + z ≤ y + H ≤ limit` (do `{H, y}` hợp lệ). Số cặp không đổi.
  - `H` ghép với `y`, `L` chưa ghép: thay `{H, y}` bằng `{H, L}` (hợp lệ), số cặp không đổi.
  - `L` ghép với `z`, `H` chưa ghép: thay `{L, z}` bằng `{L, H}`, số cặp không đổi.
  - Cả hai chưa ghép: thêm `{L, H}` làm tăng số cặp, mâu thuẫn với tính tối đa.

  Vậy ghép `(L, H)` rồi bỏ cả hai không làm mất tính tối ưu; quy nạp theo số người còn lại. Đó chính là thuật toán hai con trỏ (`lo` là `L`, `hi` là `H`).

**(c) Cách 4: `can(m)` ngoài–trong và đơn điệu.** Nếu tồn tại `m` cặp rời nhau hợp lệ trên tập `X` gồm `2m` người, sắp `x_0 ≤ … ≤ x_{2m−1}`, thì cặp ngoài–trong `(x_i, x_{2m−1−i})` cũng hợp lệ. Thật vậy, giả sử `x_i + x_{2m−1−i} > limit` với `i < m`: mọi phần tử chỉ số `≥ 2m−1−i` (có `i + 1` phần tử, giá trị `≥ x_{2m−1−i}`) không thể ghép với phần tử nào chỉ số `≥ i`; bởi vậy phần tử của nhóm này chỉ ghép được với các phần tử chỉ số `< i` (chỉ `i` phần tử): `i + 1` phần tử không thể ghép đôi rời nhau với `i` đối tác. Mâu thuẫn. Thay `X` bằng `2m` người nhẹ nhất (từng phần tử không nặng hơn) vẫn hợp lệ. Vậy `can(m)` đúng ⇔ có `m` cặp, và hiển nhiên nếu có `m` cặp thì có `m − 1` cặp: `can` đơn điệu, nhị phân tìm được `m*`.

**(d) Cách 3** thực hiện đúng quy tắc của Cách 2 nhưng gộp các người cùng cân nặng.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **vét cạn mọi phân hoạch** trên mọi đầu vào nhỏ (mục 12D) và khớp nhau trên dữ liệu lớn (12E–F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. DP bitmask | `O(2^n · n)` | `O(2^n)` | oracle, `n ≤ ~14` |
| 2. Sort + hai con trỏ | `O(n log n)` | `O(1)`–`O(n)` | **đáp án chính** |
| 3. Counting sort | `O(n + limit)` | `O(limit)` | khi `limit` nhỏ |
| 4. Nhị phân trên số cặp | `O(n log n)` | `O(n)` | góc nhìn khác |

Đo thực tế (`n = 5·10^4`, `limit = 3·10^4`):

In [15]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(881)
limit_big = 30000
big = [rng.randint(1, limit_big) for _ in range(50000)]
for name, fn in [("two_pointers", boats_two_pointers), ("counting", boats_counting), ("binary", boats_binary_search)]:
    print(f"n = 5e4 {name:13s} {bench(fn, (big, limit_big)) * 1e3:8.2f} ms")
small = [rng.randint(1, 20) for _ in range(14)]
print(f"bitmask n = 14: {bench(boats_bitmask, (small, 20), 1) * 1e3:8.1f} ms")

n = 5e4 two_pointers     15.87 ms
n = 5e4 counting          9.55 ms


n = 5e4 binary           66.62 ms
bitmask n = 14:      0.5 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi đầu vào nhỏ với oracle vét cạn phân hoạch, (E) ngẫu nhiên có seed, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(people, limit):
    """Vét cạn MỌI phân hoạch thành nhóm 1 hoặc 2 người hợp lệ (đệ quy trên người đầu tiên); trả số thuyền ít nhất."""
    n = len(people)
    best = [n]

    def rec(remaining, boats):
        if boats >= best[0]:
            return
        if not remaining:
            best[0] = boats
            return
        first, rest = remaining[0], remaining[1:]
        rec(rest, boats + 1)                              # first đi một mình
        for k, other in enumerate(rest):                  # hoặc ghép với từng người khác
            if first + other <= limit:
                rec(rest[:k] + rest[k + 1:], boats + 1)

    rec(list(people), 0)
    return best[0]


IMPLS = {
    "bitmask": boats_bitmask,
    "two_pointers": boats_two_pointers,
    "counting": boats_counting,
    "binary": boats_binary_search,
    "Solution": lambda p, lim: Solution().numRescueBoats(list(p), lim),
    "SolutionBitmask": lambda p, lim: SolutionBitmask().numRescueBoats(list(p), lim),
    "SolutionTwoPointers": lambda p, lim: SolutionTwoPointers().numRescueBoats(list(p), lim),
    "SolutionCounting": lambda p, lim: SolutionCounting().numRescueBoats(list(p), lim),
    "SolutionBinary": lambda p, lim: SolutionBinary().numRescueBoats(list(p), lim),
}


def check(people, limit, want=None, names=None):
    if want is None:
        want = oracle(people, limit)
    for name in (names or IMPLS):
        got = IMPLS[name](list(people), limit)
        assert got == want, (name, people, limit, got, want)


for (p, lim), want in examples:
    check(p, lim, want)
    assert oracle(p, lim) == want
print("ví dụ đề: oracle vét cạn phân hoạch và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle vét cạn phân hoạch và 9 cài đặt đúng


### B. Edge case

- Một người: `1` thuyền.
- Hai người ghép được / không ghép được.
- Mọi người nhẹ (`2·max ≤ limit`): `⌈n/2⌉` thuyền.
- Mọi người nặng bằng `limit`: `n` thuyền.
- Số người **lẻ** (người đứng giữa).
- Cân nặng bằng nhau, bằng `limit/2` (ghép từng cặp).
- Một người nặng `limit` và nhiều người nhẹ.

In [17]:
cases = [
    ([1], 1, 1), ([1], 5, 1), ([5], 5, 1), ([1, 1], 2, 1), ([1, 2], 2, 2), ([3, 3], 5, 2),
    ([1, 1, 1, 1, 1], 10, 3), ([4, 4, 4, 4], 8, 2), ([5, 5, 5], 5, 3), ([2, 2, 2, 2, 2, 2, 2], 4, 4),
    ([1, 1, 1, 10], 10, 3), ([10, 1, 2, 3], 10, 3), ([3, 5, 3, 4], 5, 4), ([1, 2, 3, 4, 5], 6, 3),
]
for p, lim, want in cases:
    check(p, lim, want)
print(len(cases), "edge case: đúng")

14 edge case: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_adjacent(people, limit):                       # sắp xếp rồi ghép các phần tử liền kề
    p = sorted(people)
    boats, i = 0, 0
    while i < len(p):
        if i + 1 < len(p) and p[i] + p[i + 1] <= limit:
            i += 2
        else:
            i += 1
        boats += 1
    return boats


def bug_no_sort(people, limit):                        # không sắp xếp, hai con trỏ trên mảng gốc
    p = list(people)
    lo, hi, boats = 0, len(p) - 1, 0
    while lo <= hi:
        if p[lo] + p[hi] <= limit:
            lo += 1
        hi -= 1
        boats += 1
    return boats


def bug_lo_lt_hi(people, limit):                       # while lo < hi: bỏ sót người đứng giữa
    p = sorted(people)
    lo, hi, boats = 0, len(p) - 1, 0
    while lo < hi:
        if p[lo] + p[hi] <= limit:
            lo += 1
        hi -= 1
        boats += 1
    return boats


def bug_strict(people, limit):                         # < limit thay vì <= limit
    p = sorted(people)
    lo, hi, boats = 0, len(p) - 1, 0
    while lo <= hi:
        if p[lo] + p[hi] < limit:
            lo += 1
        hi -= 1
        boats += 1
    return boats


def bug_three_per_boat(people, limit):                 # cho phép 3 người một thuyền (greedy nặng nhất + nhẹ nhất liên tiếp)
    p = sorted(people)
    lo, hi, boats = 0, len(p) - 1, 0
    while lo <= hi:
        cap = limit - p[hi]
        hi -= 1
        while lo <= hi and p[lo] <= cap:               # nhét cho tới khi hết chỗ
            cap -= p[lo]
            lo += 1
        boats += 1
    return boats


def bug_light_first(people, limit):                    # ghép người nhẹ nhất với người nặng nhất NẾU nặng nhất vừa; nếu không thì nhẹ nhất đi một mình
    p = sorted(people)
    lo, hi, boats = 0, len(p) - 1, 0
    while lo <= hi:
        if p[lo] + p[hi] <= limit:
            hi -= 1
        lo += 1                                        # SAI: luôn đưa người nhẹ nhất đi, bỏ hi lại
        boats += 1
    return boats


BUGS = [bug_adjacent, bug_no_sort, bug_lo_lt_hi, bug_strict, bug_three_per_boat, bug_light_first]
rng = random.Random(3)
tests = []
for _ in range(800):
    lim = rng.randint(2, 12)
    tests.append(([rng.randint(1, lim) for _ in range(rng.randint(1, 7))], lim))
for bug in BUGS:
    bad = next(((p, lim) for p, lim in tests if bug(p, lim) != Solution().numRescueBoats(list(p), lim)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên people = {bad[0]}, limit = {bad[1]}: ra {bug(*bad)}, đúng {Solution().numRescueBoats(list(bad[0]), bad[1])}")

bug_adjacent         sai trên people = [2, 10, 1, 8, 5], limit = 12: ra 4, đúng 3
bug_no_sort          sai trên people = [5, 2, 3, 5, 4], limit = 5: ra 5, đúng 4
bug_lo_lt_hi         sai trên people = [2, 10, 1, 8, 5], limit = 12: ra 2, đúng 3
bug_strict           sai trên people = [5, 2, 3, 5, 4], limit = 5: ra 5, đúng 4
bug_three_per_boat   sai trên people = [1, 2, 3, 1, 5, 1], limit = 8: ra 2, đúng 3
bug_light_first      sai trên people = [5, 2, 3, 5, 4], limit = 5: ra 5, đúng 4


### D. Vét cạn mọi đầu vào nhỏ

Mọi `limit` từ `1` tới `5`, mọi mảng độ dài `1..6` có giá trị `1..limit` (kể cả có lặp và mọi thứ tự). Oracle: vét cạn phân hoạch.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("bitmask", "two_pointers", "counting", "binary", "Solution")
for lim in range(1, 6):
    for n in range(1, 7):
        for p in itertools.product(range(1, lim + 1), repeat=n):
            want = oracle(p, lim)
            for name in fast:
                assert IMPLS[name](list(p), lim) == want, (name, p, lim)
            cnt += 1
print(cnt, "bộ (people, limit): 5 cài đặt khớp oracle vét cạn phân hoạch;", round(time.perf_counter() - t0, 1), "s")

26214 bộ (people, limit): 5 cài đặt khớp oracle vét cạn phân hoạch; 1.0 s


### E. Ngẫu nhiên có seed và tính chất

Cỡ vừa: so các cách nhanh với DP bitmask (`n ≤ 12`). Thêm **tính chất**: (1) kết quả không phụ thuộc thứ tự đầu vào; (2) thêm một người làm số thuyền tăng `0` hoặc `1`; (3) tăng `limit` không làm tăng số thuyền; (4) `⌈n/2⌉ ≤ đáp án ≤ n`.

In [20]:
rng = random.Random(20240881)
for _ in range(1500):
    lim = rng.randint(1, 30)
    n = rng.randint(1, 12)
    p = [rng.randint(1, lim) for _ in range(n)]
    want = boats_bitmask(p, lim)
    for name in ("two_pointers", "counting", "binary", "Solution", "SolutionCounting", "SolutionBinary", "SolutionTwoPointers"):
        assert IMPLS[name](list(p), lim) == want, (name, p, lim)
    shuffled = p[:]
    rng.shuffle(shuffled)
    assert Solution().numRescueBoats(shuffled, lim) == want                        # (1) thứ tự không quan trọng
    extra = rng.randint(1, lim)
    assert want <= Solution().numRescueBoats(p + [extra], lim) <= want + 1         # (2) thêm một người
    assert Solution().numRescueBoats(list(p), lim + rng.randint(0, 10)) <= want    # (3) limit lớn hơn
    assert (n + 1) // 2 <= want <= n                                                # (4)
print("1500 ca ngẫu nhiên: khớp DP bitmask; 4 tính chất đều đúng")

1500 ca ngẫu nhiên: khớp DP bitmask; 4 tính chất đều đúng


### F. Giới hạn

`n = 5·10^4`, `limit = 3·10^4`: các cách nhanh khớp nhau; kiểm tra trường hợp cực đoan (`n` lẻ, mọi người nhẹ, mọi người `limit`) bằng công thức đóng.

In [21]:
rng = random.Random(500)
fast = ["two_pointers", "counting", "binary", "Solution", "SolutionTwoPointers", "SolutionCounting", "SolutionBinary"]
for lim in (30000, 20000, 10, 1):
    for kind in ("uniform", "light", "heavy"):
        n = 50000
        if kind == "uniform":
            p = [rng.randint(1, lim) for _ in range(n)]
        elif kind == "light":
            p = [rng.randint(1, max(1, lim // 2)) for _ in range(n)]
        else:
            p = [rng.randint(max(1, lim // 2 + 1), lim) for _ in range(n)]
        res = {name: IMPLS[name](list(p), lim) for name in fast}
        assert len(set(res.values())) == 1, (lim, kind, res)
print("limit tới 3·10^4, n = 5·10^4: 7 cài đặt cùng đáp án trên 12 bộ dữ liệu")
n = 49999
assert Solution().numRescueBoats([1] * n, 2) == (n + 1) // 2                      # ghép từng cặp, n lẻ
assert Solution().numRescueBoats([30000] * 50000, 30000) == 50000                 # không ai ghép được
assert Solution().numRescueBoats([15000] * 50000, 30000) == 25000                 # ghép đúng limit
assert Solution().numRescueBoats([1] * 49999 + [30000], 30000) == 25001                 # 30000 đi một mình + 25000 thuyền cho 49999 người nhẹ
print("công thức đóng: đúng")

limit tới 3·10^4, n = 5·10^4: 7 cài đặt cùng đáp án trên 12 bộ dữ liệu
công thức đóng: đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: ghép các phần tử liền kề sau khi sắp xếp

In [22]:
p, lim = [1, 2, 2, 3], 3
print("liền kề:", bug_adjacent(p, lim), "| đúng:", Solution().numRescueBoats(list(p), lim))
bad = next((q, l) for q, l in tests if bug_adjacent(q, l) != Solution().numRescueBoats(list(q), l))
print("phản ví dụ:", bad, "->", bug_adjacent(*bad), "| đúng:", Solution().numRescueBoats(list(bad[0]), bad[1]))

liền kề: 3 | đúng: 3
phản ví dụ: ([2, 10, 1, 8, 5], 12) -> 4 | đúng: 3


**Vì sao sai:** ghép `(1, 2)` rồi `(2, 3)` không được — nhưng ghép `(1, 3)`/`(2, 2)` có thể được. **Đúng:** ghép `(nhẹ nhất, nặng nhất)`.

### Lỗi 2: quên sắp xếp

In [23]:
bad = next((q, l) for q, l in tests if bug_no_sort(q, l) != Solution().numRescueBoats(list(q), l))
print(bad, "-> không sort:", bug_no_sort(*bad), "| đúng:", Solution().numRescueBoats(list(bad[0]), bad[1]))

([5, 2, 3, 5, 4], 5) -> không sort: 5 | đúng: 4


**Vì sao sai:** hai con trỏ chỉ có nghĩa trên mảng đã sắp xếp. **Đúng:** `people.sort()` trước.

### Lỗi 3: `while lo < hi` bỏ sót người đứng giữa

In [24]:
p, lim = [3, 5, 3, 4], 5
print("lo < hi:", bug_lo_lt_hi(p, lim), "| đúng:", Solution().numRescueBoats(list(p), lim))
assert bug_lo_lt_hi([5], 5) == 0 and Solution().numRescueBoats([5], 5) == 1

lo < hi: 3 | đúng: 4


**Vì sao sai:** khi `lo == hi` vẫn còn **một người** chưa lên thuyền. **Đúng:** `while lo <= hi`.

### Lỗi 4: `<` thay vì `≤`

In [25]:
print("strict:", bug_strict([1, 2], 3), "| đúng:", Solution().numRescueBoats([1, 2], 3))
assert bug_strict([1, 2], 3) == 2

strict: 2 | đúng: 1


**Vì sao sai:** tổng đúng bằng `limit` vẫn hợp lệ. **Đúng:** `<= limit`.

### Lỗi 5: cho phép 3 người một thuyền

In [26]:
print("3 người:", bug_three_per_boat([1, 1, 1], 3), "| đúng:", Solution().numRescueBoats([1, 1, 1], 3))
assert bug_three_per_boat([1, 1, 1], 3) == 1 and Solution().numRescueBoats([1, 1, 1], 3) == 2

3 người: 1 | đúng: 2


**Vì sao sai:** đề giới hạn **tối đa hai** người. **Đúng:** mỗi bước chỉ thêm tối đa một người nhẹ.

### Lỗi 6 (bonus): đưa người nhẹ nhất đi, bỏ người nặng lại

In [27]:
bad = next((q, l) for q, l in tests if bug_light_first(q, l) != Solution().numRescueBoats(list(q), l))
print(bad, "->", bug_light_first(*bad), "| đúng:", Solution().numRescueBoats(list(bad[0]), bad[1]))

([5, 2, 3, 5, 4], 5) -> 5 | đúng: 4


**Vì sao sai:** người nặng nhất là ràng buộc chặt nhất, phải xử lý trước. **Đúng:** `hi -= 1` luôn luôn, `lo += 1` khi ghép được.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "ghép tối đa hai phần tử một nhóm với giới hạn tổng, tối thiểu số nhóm" → **sắp xếp + hai con trỏ**, ghép **nhẹ nhất với nặng nhất** (cặp ngoài–trong).

**Mẹo:**

1. Số nhóm `= n − số cặp`; tối đa hoá số cặp.
2. Phần tử **nặng nhất** là ràng buộc chặt nhất: xử lý nó trước.
3. Khi `lo == hi` vẫn còn một phần tử: dùng `lo <= hi`.
4. Cân nặng nhỏ (`≤ 3·10^4`): counting sort bỏ `log n`.
5. Chứng minh tham lam bằng đổi chỗ hoặc `can(m)` đơn điệu; kiểm bằng vét cạn.

**Họ bài cùng pattern:** Assign Cookies (455), Two Sum II (167), Container With Most Water (11), 3Sum (15), Minimum Number of Moves to Seat Everyone (2037), Maximum Number of Pairs in Array (2341), Bag of Tokens (948).

## 15. Biến thể

1. **Chở tối đa `k` người một thuyền** (bin packing: NP-khó): chỉ có cách xấp xỉ.
2. **Xuất danh sách các thuyền** (kế hoạch), không chỉ số lượng.
3. **Số cặp tối đa** (không cần số thuyền).
4. **Thuyền có sức chở khác nhau.**
5. **Assign Cookies (455)** — cũng tham lam với hai con trỏ.

Cài biến thể 2 (kế hoạch, kèm hàm kiểm tra hợp lệ) và 3, đối chiếu oracle:

In [28]:
def boats_plan(people, limit):
    """Trả danh sách các thuyền (mỗi thuyền là danh sách chỉ số người), dùng sort + hai con trỏ."""
    order = sorted(range(len(people)), key=lambda i: people[i])
    lo, hi, plan = 0, len(order) - 1, []
    while lo <= hi:
        if lo != hi and people[order[lo]] + people[order[hi]] <= limit:
            plan.append([order[lo], order[hi]])
            lo += 1
        else:
            plan.append([order[hi]])
        hi -= 1
    return plan


def valid_plan(people, limit, plan):
    seen = sorted(i for boat in plan for i in boat)
    return seen == list(range(len(people))) and all(1 <= len(b) <= 2 and sum(people[i] for i in b) <= limit for b in plan)


def max_pairs(people, limit):
    return len(people) - Solution().numRescueBoats(list(people), limit)


rng = random.Random(15)
for _ in range(500):
    lim = rng.randint(2, 15)
    p = [rng.randint(1, lim) for _ in range(rng.randint(1, 8))]
    plan = boats_plan(p, lim)
    assert valid_plan(p, lim, plan) and len(plan) == oracle(p, lim)             # kế hoạch hợp lệ và tối ưu
    assert max_pairs(p, lim) == sum(1 for b in plan if len(b) == 2)
print("boats_plan: 500 kế hoạch hợp lệ (mỗi người đúng một thuyền, ≤ 2 người, ≤ limit) và tối ưu; max_pairs khớp")

boats_plan: 500 kế hoạch hợp lệ (mỗi người đúng một thuyền, ≤ 2 người, ≤ limit) và tối ưu; max_pairs khớp


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP bitmask | thử mọi cách ghép | `O(2^n · n)` | `O(2^n)` | oracle, `n` nhỏ |
| 2. Sort + hai con trỏ | ghép nhẹ nhất với nặng nhất | `O(n log n)` | `O(1)` | **đáp án chính** |
| 3. Counting sort | hai con trỏ trên bảng đếm | `O(n + limit)` | `O(limit)` | `limit` nhỏ |
| 4. Nhị phân số cặp | `can(m)` ngoài–trong | `O(n log n)` | `O(n)` | góc nhìn khác |

Cả bốn khớp oracle vét cạn trên mọi đầu vào nhỏ và khớp nhau trên `n = 5·10^4` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().numRescueBoats([2, 2, 2, 2], 4), Solution().numRescueBoats([1, 2, 3, 4, 5], 5), Solution().numRescueBoats([5, 1, 4, 2], 6), max_pairs([5, 1, 4, 2], 6))

2 3 2 2


### Bài 2 (tính tay)

`people = [1, 3, 4, 5, 5, 7]`, `limit = 8`. (a) Sắp xếp rồi chạy hai con trỏ, ghi từng bước. (b) Số thuyền? (c) Số cặp tối đa? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai con trỏ:

In [30]:
def two_pointers_todo(people, limit):
    people = sorted(people)
    lo, hi, boats = 0, len(people) - 1, 0
    while lo <= hi:
        pass            # TODO 1: nếu people[lo] + people[hi] <= limit thì lo += 1
        pass            # TODO 2: hi -= 1 và boats += 1
    return boats


# Khi làm xong, bỏ comment để kiểm tra:
# assert [two_pointers_todo(p, lim) for (p, lim), _ in examples] == [1, 3, 4]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **counting sort**:

In [31]:
class MySolution:
    def numRescueBoats(self, people, limit):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().numRescueBoats(list(p), lim) == w for (p, lim), w in examples)

### Bài 5 (mini challenge): Assign Cookies (455)

Mỗi đứa trẻ `g[i]` cần cái bánh có kích cỡ `≥ g[i]`; mỗi bánh `s[j]` cho tối đa một trẻ. Viết `find_content_children(g, s)` trả số trẻ tối đa được thoả mãn (sắp xếp + hai con trỏ), kiểm tra bằng vét cạn.

In [32]:
def find_content_children(g, s):
    pass            # TODO


# Khi làm xong:
# assert find_content_children([1, 2, 3], [1, 1]) == 1 and find_content_children([1, 2], [1, 2, 3]) == 2

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: sắp xếp, hai con trỏ.
- Mức 2: `[2,2,2,2]`, `4`: mỗi cặp `2 + 2 = 4` → `2` thuyền; `[1,2,3,4,5]`, `5`: `(1,5)`? `6 > 5` ✗ → `5` đi một mình; `(1,4)` ✓; `(2,3)` ✓ → `3` thuyền.
- Mức 3: `[5,1,4,2]`, `6`: sorted `[1,2,4,5]`: `(1,5)` ✓, `(2,4)` ✓ → `2` thuyền, `2` cặp. Kết quả `2 3 2 2`.

**Bài 2**

- Mức 1: sorted `[1,3,4,5,5,7]`.
- Mức 2: `(1,7)` ✓ (tổng 8); `(3,5)` ✓ (tổng 8); còn `[4, 5]`: `4 + 5 = 9 > 8` nên `5` đi một mình; cuối cùng `4` (khi `lo == hi`) đi một mình.
- Mức 3: các thuyền `(1,7)`, `(3,5)`, `(5)`, `(4)` → `4` thuyền, `2` cặp.

**Bài 3**

- Mức 1: `if p[lo] + p[hi] <= limit: lo += 1`.
- Mức 2: luôn `hi -= 1`.
- Mức 3: `boats += 1` mỗi vòng.

**Bài 4**

- Mức 1: `cnt = [0] * (limit + 1)`.
- Mức 2: hai con trỏ trên `cnt`, ghép `min(cnt[lo], cnt[hi])` cặp.
- Mức 3: trường hợp `lo == hi` xử lý riêng.

**Bài 5**

- Mức 1: sắp xếp `g`, `s`.
- Mức 2: hai con trỏ `i` (trẻ), `j` (bánh); bánh `s[j] ≥ g[i]` thì thoả mãn trẻ `i`.
- Mức 3: luôn `j += 1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
assert (Solution().numRescueBoats([2, 2, 2, 2], 4), Solution().numRescueBoats([1, 2, 3, 4, 5], 5), Solution().numRescueBoats([5, 1, 4, 2], 6), max_pairs([5, 1, 4, 2], 6)) == (2, 3, 2, 2)
assert oracle([1, 2, 3, 4, 5], 5) == 3
print("Bài 1 OK: 2 3 2 2")

Bài 1 OK: 2 3 2 2


### Bài 2

In [34]:
p, lim = [1, 3, 4, 5, 5, 7], 8
trace_two_pointers(p, lim)
assert Solution().numRescueBoats(list(p), lim) == oracle(p, lim)
print("số cặp tối đa:", max_pairs(p, lim))
print("Bài 2 OK")

sau sắp xếp: [1, 3, 4, 5, 5, 7], limit = 8
 lo  hi  p[lo]+p[hi]  quyết định                     boats
  0   5            8  ghép (1,7)                     1
  1   4            8  ghép (3,5)                     2
  2   3            9  5 đi một mình                  3
  2   2            8  4 đi một mình                  4
=> số thuyền = 4
số cặp tối đa: 2
Bài 2 OK


### Bài 3

In [35]:
def two_pointers_answer(people, limit):
    people = sorted(people)
    lo, hi, boats = 0, len(people) - 1, 0
    while lo <= hi:
        if people[lo] + people[hi] <= limit:
            lo += 1
        hi -= 1
        boats += 1
    return boats


assert [two_pointers_answer(p, lim) for (p, lim), _ in examples] == [1, 3, 4]
for p, lim in tests[:300]:
    assert two_pointers_answer(p, lim) == oracle(p, lim)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def numRescueBoats(self, people, limit):
        cnt = [0] * (limit + 1)
        for w in people:
            cnt[w] += 1
        lo, hi, boats = 1, limit, 0
        while True:
            while lo <= hi and cnt[lo] == 0:
                lo += 1
            while hi >= lo and cnt[hi] == 0:
                hi -= 1
            if lo > hi:
                return boats
            if lo == hi:
                return boats + ((cnt[lo] + 1) // 2 if 2 * lo <= limit else cnt[lo])
            if lo + hi <= limit:
                k = min(cnt[lo], cnt[hi])
                boats += k
                cnt[lo] -= k
                cnt[hi] -= k
            else:
                boats += cnt[hi]
                cnt[hi] = 0


assert all(MySolutionAnswer().numRescueBoats(list(p), lim) == w for (p, lim), w in examples)
for p, lim in tests[:300]:
    assert MySolutionAnswer().numRescueBoats(list(p), lim) == oracle(p, lim)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def find_content_children_answer(g, s):
    g, s = sorted(g), sorted(s)
    i = 0
    for j in range(len(s)):
        if i < len(g) and s[j] >= g[i]:
            i += 1
    return i


def children_brute(g, s):
    best = 0
    # vét cạn: thử mọi tập con trẻ và mọi cách gán bánh
    n, m = len(g), len(s)
    for mask in range(1 << n):
        kids = [g[i] for i in range(n) if mask >> i & 1]
        if len(kids) > m:
            continue
        for perm in itertools.permutations(range(m), len(kids)):
            if all(s[perm[t]] >= kids[t] for t in range(len(kids))):
                best = max(best, len(kids))
                break
    return best


assert find_content_children_answer([1, 2, 3], [1, 1]) == 1 and find_content_children_answer([1, 2], [1, 2, 3]) == 2
rng = random.Random(5)
for _ in range(300):
    g = [rng.randint(1, 6) for _ in range(rng.randint(0, 5))]
    s = [rng.randint(1, 6) for _ in range(rng.randint(0, 5))]
    assert find_content_children_answer(g, s) == children_brute(g, s)
print("Bài 5 OK: Assign Cookies khớp vét cạn")

Bài 5 OK: Assign Cookies khớp vét cạn


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (nhóm tối đa 2 phần tử với giới hạn tổng, tối thiểu số nhóm):**

```python
class Solution:
    def numRescueBoats(self, people: List[int], limit: int) -> int:
        people.sort()
        lo, hi, boats = 0, len(people) - 1, 0
        while lo <= hi:
            if people[lo] + people[hi] <= limit:   # nhẹ nhất ghép được với nặng nhất
                lo += 1
            hi -= 1                                # nặng nhất luôn lên thuyền
            boats += 1
        return boats
```

**Checklist trước khi submit:**

- [ ] Đã **sắp xếp** trước khi dùng hai con trỏ?
- [ ] Ghép **nhẹ nhất với nặng nhất** (không phải liền kề)?
- [ ] `while lo <= hi` (không bỏ sót người đứng giữa)?
- [ ] Điều kiện `<= limit` (không `<`)?
- [ ] Mỗi thuyền tối đa **hai** người?
- [ ] Đã thử `[1,2],3 → 1`, `[3,2,2,1],3 → 3`, `[3,5,3,4],5 → 4`, một người, `n` lẻ?
- [ ] Đúng tên `class Solution`, `numRescueBoats(self, people, limit)`, không in debug.